In [0]:
!pip install -r ../requirements.txt
%restart_python

In [0]:
%sql
-- Enable Change Data Feed (required for Delta Sync indexes)
ALTER TABLE clinton_emails.silver.email_categories 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Verify it's enabled
DESCRIBE DETAIL clinton_emails.silver.email_categories;

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingVectorColumn,
    VectorIndexType,
    PipelineType
)
import traceback

w = WorkspaceClient()

# Define index configuration
endpoint_name = "standard_demo_api_endpoint"
index_name = "clinton_emails.silver.email_categories_index"
source_table = "clinton_emails.silver.email_categories"

print(f"Creating Delta Sync index: {index_name}")
print(f"Source table: {source_table}")
print(f"Endpoint: {endpoint_name}")
print(f"Embedding dimension: 3072\n")

try:
    # Create the index
    print("Building index specification...")
    spec = DeltaSyncVectorIndexSpecRequest(
        source_table=source_table,
        embedding_vector_columns=[
            EmbeddingVectorColumn(
                name="embedding",
                embedding_dimension=3072
            )
        ],
        pipeline_type=PipelineType.TRIGGERED
    )
    print(f"Spec created successfully")
    
    print("\nCalling create_index API...")
    index = w.vector_search_indexes.create_index(
        name=index_name,
        endpoint_name=endpoint_name,
        primary_key="id",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=spec
    )
    
    print("✅ Index creation initiated successfully!")
    print(f"\nIndex details:")
    print(f"  Name: {index.name}")
    if index.status:
        print(f"  Ready: {index.status.ready}")
        print(f"  Message: {index.status.message if hasattr(index.status, 'message') else 'N/A'}")
    print(f"\nThe index will now perform the initial sync.")
    print(f"You can monitor progress in the UI or by checking the index status.")
    
except Exception as e:
    print(f"❌ Error creating index: {e}")
    print(f"\nError type: {type(e).__name__}")
    print(f"\nFull traceback:")
    traceback.print_exc()
    if hasattr(e, 'error_code'):
        print(f"\nError code: {e.error_code}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

index_name = "clinton_emails.silver.email_categories_index"

print(f"Checking status for index: {index_name}\n")

try:
    index = w.vector_search_indexes.get_index(index_name=index_name)
    
    print(f"Index Name: {index.name}")
    print(f"Primary Key: {index.primary_key}")
    print(f"Index Type: {index.index_type}")
    
    if index.status:
        print(f"\nStatus:")
        print(f"  Ready: {index.status.ready}")
        if hasattr(index.status, 'indexed_row_count'):
            print(f"  Indexed Rows: {index.status.indexed_row_count}")
        if hasattr(index.status, 'message'):
            print(f"  Message: {index.status.message}")
    
    if index.delta_sync_index_spec:
        print(f"\nDelta Sync Configuration:")
        print(f"  Source Table: {index.delta_sync_index_spec.source_table}")
        print(f"  Pipeline Type: {index.delta_sync_index_spec.pipeline_type}")
        if index.delta_sync_index_spec.embedding_vector_columns:
            for col in index.delta_sync_index_spec.embedding_vector_columns:
                print(f"  Embedding Column: {col.name} (dimension: {col.embedding_dimension})")
    
    print(f"\n✅ Index is configured and {'ready' if index.status and index.status.ready else 'syncing'}")
    
except Exception as e:
    print(f"❌ Error checking index: {e}")

In [0]:
from databricks.ai_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient

# Check if index is ready first
w = WorkspaceClient()
index_name = "clinton_emails.silver.email_categories_index"

index_info = w.vector_search_indexes.get_index(index_name=index_name)

if not index_info.status.ready:
    print("⏳ Index is not ready yet!")
    print(f"Status: {index_info.status.message}")
    print("\nPlease wait a few minutes and rerun this cell.")
    print("Run Cell 3 (Check index status) to monitor progress.")
else:
    print("✅ Index is ready! Performing similarity search...\n")
    
    # Use VectorSearchClient for querying
    vsc = VectorSearchClient()
    
    # For demonstration, let's use the embedding from one of the categories
    # In practice, you'd generate a query embedding for your search term
    df = spark.table("clinton_emails.silver.email_categories").limit(1)
    query_embedding = df.select("embedding").first()["embedding"]
    
    print(f"Searching with embedding vector (dimension {len(query_embedding)})...\n")
    
    # Perform similarity search
    results = vsc.get_index(
        endpoint_name="standard_demo_api_endpoint",
        index_name=index_name
    ).similarity_search(
        query_vector=query_embedding,
        columns=["id", "category", "description"],
        num_results=5
    )
    
    print("Top 5 similar categories:")
    display(results)